# GLiNER2 한국어 추출: 단어 분할기

[언어와 체크포인트 비교](01_language_and_checkpoint.ipynb)에서 GLiNER가 `GCP에서`처럼 조사를 엔티티 범위에 포함하고, 조사를 띄어 쓰면 정확한 범위를 찾는다는 것을 확인했다.
`gliner2`는 모델 입력 전에 문장을 단어로 나누는 분할기(word splitter)를 교체할 수 있다(`set_word_splitter`). 본 실험은 분할기를 공백·문자·형태소 단위로 바꾸었을 때 한국어 엔티티·관계 추출이 얼마나 개선되는지, 그 결과가 LLM 추출과 비교해 어느 수준인지를 확인한다.

| 분할기 | 단어 경계 | 예: `삼성전자는 엔비디아와` |
|---|---|---|
| `whitespace` | 공백과 문장부호 (공개 체크포인트의 학습 조건, 기본값) | `삼성전자는` / `엔비디아와` |
| `char` | 한글은 한 글자씩, 영문·숫자 연속은 한 단어 (`gliner2` 내장) | `삼` / `성` / `전` / `자` / `는` / … |
| `kiwi` | 한국어 형태소 분석기 [Kiwi](https://github.com/bab2min/kiwipiepy)의 형태소 | `삼성전자` / `는` / `엔비디아` / `와` |

## 결과 요약

> 아래 수치는 2026-11-10 실행 결과(`korean_word_splitter.json`, 문장 9개 × 3회)이다. 엔티티 정답 21개, 관계 정답 9개.

| 조건 | 엔티티 정확 일치 | 조사 포함 | 타입 오류 | 관계 적중 | 생성 관계 수 | 중앙값 지연 |
|---|---|---|---|---|---|---|
| `base_whitespace` (기본 설정) | 1/21 | 11 | 3 | 2/9 | 5 | 119 ms |
| `multi_whitespace` | 5/21 | 13 | 0 | (8/9)* | 17 | 138 ms |
| **`multi_kiwi`** | **16/21** | **0** | **0** | 8/9 | 16 | 139 ms |
| `llm_only` | 19.7/21 | 0 | 0 | 8/9 | 17.7 | 1,836 ms |

\* 조사가 붙은 이름(`삼성전자는 -PARTNER_WITH→ 엔비디아와`)을 포함 일치로 인정한 값이다. 그래프에서는 `삼성전자`와 별개의 노드가 되므로 실질적인 적중으로 볼 수 없다.

- **형태소 분할기는 조사 문제를 완전히 없앤다.** `multi_kiwi`는 조사 포함 0건, 타입 오류 0건이며 엔티티 정확 일치가 5 → 16으로 오른다.
- **문자 분할기는 오히려 나빠진다.** 공개 체크포인트가 공백 단위로 학습되어 한 글자 단위 입력에서는 타입 판단이 무너진다.
- **base 체크포인트는 분할기를 바꿔도 한국어에 부적합하다.** `base_kiwi`도 `네이버클라우드`, `KT`를 PERSON으로 분류한다.
- **그래도 LLM과의 격차는 남는다.** 엔티티는 16 대 19.7이고, 관계는 적중 수가 같아도 GLiNER 쪽에 문장에 없는 관계와 중복 타입이 섞인다. 틀린 관계의 confidence가 정답보다 높은 경우도 있다.

## 실험 조건

| 항목 | 값 |
|---|---|
| GLiNER | base, multi × 분할기 3종, CPU, 엔티티 threshold 0.2 |
| LLM | `.env`의 `LLM_MODEL`, 구조화 출력 |
| 스키마 | POLE+O 엔티티 5종, 관계 15종 |
| 반복 | 문장 9개 × 3회 (GLiNER는 결정적 출력 여부 확인을 겸함) |

문장 1~4는 [언어와 체크포인트 비교](01_language_and_checkpoint.ipynb)의 한국어 문장이고, 5~9는 엔티티와 관계가 분명한 한국어 문장이다.
정답이 있는 문장에는 실험자가 정한 **참고 정답**을 둔다. 일반 벤치마크가 아니며 조건 간 상대 비교를 위한 기준이다.

| # | 문장 | 참고 정답 엔티티 | 참고 정답 관계 |
|---|---|---|---|
| 1 | 이제 GCP에서 매달 나가는 비용은 Cloud DNS 영역 3개, 월 약 $0.6뿐입니다. | GCP(ORG), Cloud DNS(OBJ) | – |
| 2 | 화면과 상관없는 서버 쪽 작업(공지 저장, 작업실 정보)은 이미 같이 진행하고 있어요. | – | – |
| 3 | 11월 사용량을 서비스별로 조회해 보니 금액이 붙은 항목이 하나도 없습니다. | – | – |
| 4 | 오라클 공식 입장: 고정 IP 기능을 처음 발표할 때 'There is no charge to use this feature'라고 밝혔습니다. | 오라클(ORG), 고정 IP 기능(OBJ) | – |
| 5 | 삼성전자는 어제 이재용 회장이 서울 서초구 사옥에서 엔비디아와 협력을 발표했다고 밝혔다. | 삼성전자(ORG), 이재용(PER), 서울 서초구(LOC), 엔비디아(ORG) | 삼성전자–엔비디아 PARTNER_WITH, 이재용→삼성전자 EMPLOYED_BY/MEMBER_OF |
| 6 | 김민수 팀장은 다음 주 부산에서 열리는 AWS 서밋에 카카오 개발자들과 함께 참석한다. | 김민수(PER), 부산(LOC), AWS 서밋(EVT), 카카오(ORG) | 김민수→AWS 서밋 PARTICIPATED_IN, AWS 서밋→부산 OCCURRED_AT |
| 7 | 네이버클라우드와 KT가 공동으로 판교에 데이터센터를 짓기로 했습니다. | 네이버클라우드(ORG), KT(ORG), 판교(LOC) | 네이버클라우드–KT PARTNER_WITH |
| 8 | 박지훈은 2023년부터 토스에서 백엔드 개발자로 일하고 있고, 지금은 성수동에 산다. | 박지훈(PER), 토스(ORG), 성수동(LOC) | 박지훈→토스 EMPLOYED_BY, 박지훈→성수동 RESIDES_AT |
| 9 | 쿠팡은 대만 법인을 자회사로 두고 있으며, 본사는 서울 송파구에 있다. | 쿠팡(ORG), 대만 법인(ORG), 서울 송파구(LOC) | 쿠팡→서울 송파구 HEADQUARTERS_AT, 대만 법인→쿠팡 SUBSIDIARY_OF |

## 준비

### 경고 출력 설정

체크포인트 로딩 시 출력되는 Transformers·PyTorch 경고를 숨긴다. 경고의 원인을 처리하지는 않는다.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("transformers").setLevel(logging.ERROR)

### 프로젝트 경로와 환경 변수

현재 작업 디렉터리에서 상위로 올라가며 `pyproject.toml`이 있는 프로젝트 루트를 찾아 모듈 검색 경로에 추가한다. 추출기와 스키마는 프로젝트의 `config` 모듈에서 가져온다.
`.env`에서 `OPENAI_API_KEY`와 LLM 모델 이름(`LLM_MODEL`)을 읽는다. LLM 기준선에만 필요하다.

In [ ]:
import os
from pathlib import Path
import sys

from dotenv import load_dotenv

PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
load_dotenv(PROJECT / ".env")
LLM_MODEL = os.getenv("LLM_MODEL", "gpt-5.4-mini")
RESULT_PATH = PROJECT / "experiments/results/korean_word_splitter.json"
print(f"✓ Project root: {PROJECT}")
print(f"✓ Result file: {RESULT_PATH.relative_to(PROJECT)}  LLM: {LLM_MODEL}")

### 실행 방식 선택

`RUN_EXPERIMENT = False`이면 저장된 결과 파일을 불러와 분석만 수행한다.
`True`로 바꾸면 GLiNER 체크포인트를 내려받아 로딩하고 LLM API를 호출하여 결과 파일을 덮어쓴다.

In [ ]:
RUN_EXPERIMENT = False
REPEATS = 3

### 보조 함수

결과 파일 저장·불러오기와 분석 표 출력에 쓰는 함수이다.

- `entity_summary()`: 엔티티별로 `이름(타입) 등장 횟수/실행 수 · 최대 confidence`를 이어 붙인다.
- `relation_summary()`: 관계별로 `출발점 -타입→ 도착점 등장 횟수/실행 수`를 이어 붙인다.
- `has_particle()`: Kiwi로 분석한 엔티티 이름의 마지막 형태소가 조사(`J*`)인지 판정한다.

In [ ]:
from collections import Counter
import json

from IPython.display import Markdown, display

from config.extractors import kiwi


def save(path, report):
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")


def load(path):
    return json.loads(path.read_text())


def show_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    lines += ["| " + " | ".join(str(cell) for cell in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def entity_summary(runs):
    counts, confidence = Counter(), {}
    for run in runs:
        for entity in run["result"]["entities"]:
            key = (entity["name"], entity["type"])
            confidence[key] = max(confidence.get(key, 0), entity["confidence"])
        counts.update({(e["name"], e["type"]) for e in run["result"]["entities"]})
    items = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    return "<br>".join(f"{n}({t}) {c}/{len(runs)} · {confidence[n, t]:.2f}"
                       for (n, t), c in items) or "–"


def relation_summary(runs):
    counts = Counter()
    for run in runs:
        counts.update({(r["source"], r["type"], r["target"]) for r in run["result"]["relations"]})
    return "<br>".join(f"{s} -{t}→ {o} {c}/{len(runs)}"
                       for (s, t, o), c in sorted(counts.items(), key=lambda i: (-i[1], i[0]))) or "–"


def has_particle(name):
    tokens = kiwi().tokenize(name)
    return bool(tokens) and tokens[-1].tag.startswith("J")

### 입력 문장과 참고 정답

엔티티는 `(이름, 타입)`, 관계는 `(출발점, 관계 타입 후보, 도착점, 방향 무관 여부)`로 표기한다. 관계 타입은 온톨로지에서 의미상 허용되는 후보를 모두 정답으로 인정한다.

In [ ]:
TEXTS = [
    "이제 GCP에서 매달 나가는 비용은 Cloud DNS 영역 3개, 월 약 $0.6뿐입니다.",
    "화면과 상관없는 서버 쪽 작업(공지 저장, 작업실 정보)은 이미 같이 진행하고 있어요.",
    "11월 사용량을 서비스별로 조회해 보니 금액이 붙은 항목이 하나도 없습니다.",
    "오라클 공식 입장: 고정 IP 기능을 처음 발표할 때 'There is no charge to use this feature'라고 밝혔습니다.",
    "삼성전자는 어제 이재용 회장이 서울 서초구 사옥에서 엔비디아와 협력을 발표했다고 밝혔다.",
    "김민수 팀장은 다음 주 부산에서 열리는 AWS 서밋에 카카오 개발자들과 함께 참석한다.",
    "네이버클라우드와 KT가 공동으로 판교에 데이터센터를 짓기로 했습니다.",
    "박지훈은 2023년부터 토스에서 백엔드 개발자로 일하고 있고, 지금은 성수동에 산다.",
    "쿠팡은 대만 법인을 자회사로 두고 있으며, 본사는 서울 송파구에 있다.",
]
GOLD_ENTITIES = {
    1: [("GCP", "ORGANIZATION"), ("Cloud DNS", "OBJECT")],
    4: [("오라클", "ORGANIZATION"), ("고정 IP 기능", "OBJECT")],
    5: [("삼성전자", "ORGANIZATION"), ("이재용", "PERSON"), ("서울 서초구", "LOCATION"),
        ("엔비디아", "ORGANIZATION")],
    6: [("김민수", "PERSON"), ("부산", "LOCATION"), ("AWS 서밋", "EVENT"), ("카카오", "ORGANIZATION")],
    7: [("네이버클라우드", "ORGANIZATION"), ("KT", "ORGANIZATION"), ("판교", "LOCATION")],
    8: [("박지훈", "PERSON"), ("토스", "ORGANIZATION"), ("성수동", "LOCATION")],
    9: [("쿠팡", "ORGANIZATION"), ("대만 법인", "ORGANIZATION"), ("서울 송파구", "LOCATION")],
}
GOLD_RELATIONS = {
    5: [("삼성전자", {"PARTNER_WITH"}, "엔비디아", True),
        ("이재용", {"EMPLOYED_BY", "MEMBER_OF"}, "삼성전자", False)],
    6: [("김민수", {"PARTICIPATED_IN"}, "AWS 서밋", False),
        ("AWS 서밋", {"OCCURRED_AT", "LOCATED_AT"}, "부산", False)],
    7: [("네이버클라우드", {"PARTNER_WITH"}, "KT", True)],
    8: [("박지훈", {"EMPLOYED_BY"}, "토스", False),
        ("박지훈", {"RESIDES_AT"}, "성수동", False)],
    9: [("쿠팡", {"HEADQUARTERS_AT", "LOCATED_AT"}, "서울 송파구", False),
        ("대만 법인", {"SUBSIDIARY_OF"}, "쿠팡", False)],
}
CHECKPOINTS = {"base": "fastino/gliner2.5-base-v1", "multi": "fastino/gliner2.5-multi-v1"}

### 형태소 분할기

[`config/extractors.py`](../config/extractors.py)의 `kiwi_splitter()`는 `gliner2`의 분할기 규약(`(text, lower)` → `(token, start, end)`, 원문 기준 문자 위치)을 따른다.
Kiwi는 축약형(예: `했` → `하` + `었`)을 같은 위치의 형태소 여러 개로 반환하므로, 이미 지나간 위치의 형태소는 건너뛰고 원문 표면형을 그대로 사용한다.

In [ ]:
from config.extractors import kiwi_splitter

SPLITTERS = {"whitespace": "whitespace", "char": "char", "kiwi": kiwi_splitter}
print([token for token, _, _ in kiwi_splitter("삼성전자는 엔비디아와 협력을 발표했다.", lower=False)])

### 추출기

[`config/extractors.py`](../config/extractors.py)의 두 추출기는 같은 POLE+O 스키마([`config/ontology.py`](../config/ontology.py))를 사용하고 같은 형태의 결과를 반환한다.

| 추출기 | 동작 |
|---|---|
| `GlinerExtractor` | `gliner2`의 JointIE 엔진에 엔티티 타입과 관계 타입(머리·꼬리 타입, 설명, threshold)을 스키마로 등록하고, 엔티티와 관계를 한 번에 디코딩한다. 엔티티 threshold는 0.2이다. |
| `LLMExtractor` | 같은 타입 정의를 지시문으로 전달하고 OpenAI 구조화 출력(JSON Schema)으로 엔티티와 관계를 받는다. 존재하지 않는 엔티티를 가리키거나 온톨로지가 허용하지 않는 타입 조합의 관계는 버린다. |

## 실험

### 반복 추출

1. GLiNER: 체크포인트마다 모델을 한 번 로딩하고, 분할기를 바꿔 가며 9개 문장을 `REPEATS`회 추출한다.
2. LLM: 9개 문장을 `REPEATS`회 추출한다.

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
from time import perf_counter

from config.extractors import GlinerExtractor, LLMExtractor


def record(report, mode, repeat, index, result, started):
    report["runs"].append({"mode": mode, "repeat": repeat, "message_index": index,
                           "elapsed_ms": round((perf_counter() - started) * 1000, 1), "result": result})


async def run_experiment(repeats):
    report = {
        "status": "running", "started_at": datetime.now(timezone.utc).isoformat(),
        "repeats": repeats, "llm_model": LLM_MODEL, "checkpoints": CHECKPOINTS,
        "versions": {p: version(p) for p in ("gliner2", "kiwipiepy", "openai")},
        "texts": TEXTS, "runs": [],
    }
    for name, model in CHECKPOINTS.items():
        extractor = GlinerExtractor(model)
        extractor.extract("Alice works at Acme in Seoul.")  # warm up
        for splitter_name, splitter in SPLITTERS.items():
            extractor.set_word_splitter(splitter)
            for repeat in range(1, repeats + 1):
                for index, text in enumerate(TEXTS, start=1):
                    started = perf_counter()
                    record(report, f"{name}_{splitter_name}", repeat, index, extractor.extract(text), started)
            print(f"✓ {name}_{splitter_name}")
    llm = LLMExtractor(LLM_MODEL)
    for repeat in range(1, repeats + 1):
        for index, text in enumerate(TEXTS, start=1):
            started = perf_counter()
            record(report, "llm_only", repeat, index, await llm.extract(text), started)
        print(f"✓ llm_only repeat {repeat}")
    report["status"] = "complete"
    report["completed_at"] = datetime.now(timezone.utc).isoformat()
    save(RESULT_PATH, report)
    return report

### 실행 또는 결과 불러오기

In [ ]:
report = await run_experiment(REPEATS) if RUN_EXPERIMENT else load(RESULT_PATH)
MODES = list(dict.fromkeys(run["mode"] for run in report["runs"]))
print(f"✓ status={report['status']} runs={len(report['runs'])} versions={report['versions']} llm={report['llm_model']}")

## 결과 분석

### 채점 기준

- **조사 포함**: 엔티티 이름을 Kiwi로 분석했을 때 마지막 형태소가 조사(`J*`)인 경우. 모든 문장에서 센다.
- **정확 일치**: 이름과 타입이 참고 정답과 같다.
- **포함 일치**: 타입이 같고, 공백을 제거한 이름에 정답 이름이 포함된다. `이재용 회장`처럼 범위가 조금 넓은 경우를 인정하지만 `GCP에서`도 인정되므로 조사 포함 지표와 함께 읽는다.
- **타입 오류**: 정답 이름을 포함하지만 타입이 다르다. 예: `오라클(PERSON)`.
- **정답 외**: 정답이 있는 문장(1, 4~9)에서 어떤 정답과도 이름이 겹치지 않는 엔티티.
- **관계 적중**: 관계 타입이 후보에 속하고, 출발점·도착점 이름이 각각 정답 이름을 포함한다. 방향 무관 관계는 반대 방향도 인정한다.

모든 지표는 반복마다 계산한 뒤 평균한다.

In [ ]:
from statistics import mean, median


def squash(name):
    return name.replace(" ", "").lower()


def score_entities(run):
    entities = run["result"]["entities"]
    gold = GOLD_ENTITIES.get(run["message_index"], [])
    names = [(squash(e["name"]), e["type"]) for e in entities]
    score = Counter(particle=sum(has_particle(e["name"]) for e in entities))
    for gold_name, gold_type in gold:
        key = squash(gold_name)
        typed = any(key in n and t == gold_type for n, t in names)
        score["exact"] += any(n == key and t == gold_type for n, t in names)
        score["contains"] += typed
        score["type_error"] += not typed and any(key in n for n, _ in names)
    if gold:
        score["extra"] = sum(not any(squash(g) in n for g, _ in gold) for n, _ in names)
    return score


def matches(relation, gold):
    source, types, target, undirected = gold
    s, t = squash(relation["source"]), squash(relation["target"])
    forward = squash(source) in s and squash(target) in t
    backward = undirected and squash(source) in t and squash(target) in s
    return relation["type"] in types and (forward or backward)


def score_relations(run):
    relations = run["result"]["relations"]
    gold = GOLD_RELATIONS.get(run["message_index"], [])
    return Counter(hit=sum(any(matches(r, g) for r in relations) for g in gold), total=len(relations))


def runs_of(mode, index=None, repeat=None):
    return [r for r in report["runs"] if r["mode"] == mode
            and (index is None or r["message_index"] == index)
            and (repeat is None or r["repeat"] == repeat)]


def per_repeat_mean(mode, scorer, key):
    value = mean(sum(scorer(r)[key] for r in runs_of(mode, repeat=rep))
                 for rep in range(1, report["repeats"] + 1))
    return f"{value:.0f}" if value == int(value) else f"{value:.1f}"


def deterministic(mode):
    outputs = {}
    for run in runs_of(mode):
        key = json.dumps(sorted((e["name"], e["type"]) for e in run["result"]["entities"]), ensure_ascii=False)
        outputs.setdefault(run["message_index"], set()).add(key)
    return "예" if all(len(v) == 1 for v in outputs.values()) else "아니오"


GOLD_ENTITY_TOTAL = sum(map(len, GOLD_ENTITIES.values()))
GOLD_RELATION_TOTAL = sum(map(len, GOLD_RELATIONS.values()))
print(f"gold entities={GOLD_ENTITY_TOTAL}, gold relations={GOLD_RELATION_TOTAL}")

### 엔티티 지표

In [ ]:
show_table(
    ["모드", f"정확 일치 /{GOLD_ENTITY_TOTAL}", f"포함 일치 /{GOLD_ENTITY_TOTAL}", "조사 포함",
     "타입 오류", "정답 외", "반복 간 동일", "중앙값 ms"],
    [[mode, *(per_repeat_mean(mode, score_entities, key)
              for key in ("exact", "contains", "particle", "type_error", "extra")),
      deterministic(mode), f"{median(r['elapsed_ms'] for r in runs_of(mode)):,.0f}"]
     for mode in MODES],
)

| 모드 | 정확 일치 /21 | 포함 일치 /21 | 조사 포함 | 타입 오류 | 정답 외 | 반복 간 동일 | 중앙값 ms |
|---|---|---|---|---|---|---|---|
| base_whitespace | 1 | 10 | 11 | 3 | 1 | 예 | 119 |
| base_char | 4 | 4 | 0 | 1 | 1 | 예 | 119 |
| base_kiwi | 7 | 11 | 0 | 4 | 3 | 예 | 115 |
| multi_whitespace | 5 | 18 | 13 | 0 | 1 | 예 | 138 |
| multi_char | 5 | 5 | 0 | 5 | 2 | 예 | 141 |
| **multi_kiwi** | **16** | **18** | **0** | **0** | 3 | 예 | 139 |
| llm_only | 19.7 | 21 | 0 | 0 | 2.3 | 아니오 | 1,836 |

**해석**

- `multi_whitespace`는 포함 일치가 18로 높지만 정확 일치는 5이다. 엔티티 위치는 대부분 찾지만 조사가 붙은 이름(`삼성전자는`, `부산에서`, `KT가`)으로 추출한다. 형태소 분할은 이 차이를 거의 그대로 정확 일치로 바꾼다(5 → 16).
- `multi_kiwi`의 남은 차이(포함 18, 정확 16)는 `이재용 회장`, `서울 서초구 사옥`처럼 직함이나 장소 명사까지 범위에 포함한 경우이다. 놓친 정답은 `Cloud DNS`, `고정 IP 기능`, `대만 법인`(→ `대만(LOCATION)`)이다.
- GLiNER는 모든 조건에서 반복 간 출력이 같다. LLM은 같은 입력에도 회차마다 결과가 달라진다.

### 문장별 엔티티

주요 조건의 엔티티를 문장별로 비교한다. `n/3`은 등장 횟수, 마지막 숫자는 최대 confidence이다.

In [ ]:
FOCUS = ["base_whitespace", "multi_whitespace", "multi_kiwi", "llm_only"]
show_table(["#", *FOCUS],
           [[index, *(entity_summary(runs_of(mode, index)) for mode in FOCUS)]
            for index in range(1, len(TEXTS) + 1)])

**해석**

- 형태소 분할 후에는 `삼성전자는` → `삼성전자`, `부산에서` → `부산`, `KT가` → `KT`, `박지훈은` → `박지훈`처럼 범위가 정리되고 confidence도 오른다(예: `네이버클라우드` 0.62 → 1.00).
- 분할기와 무관한 multi 체크포인트의 노이즈는 남는다. `11월(EVENT)`, `협력(EVENT)`, `데이터센터(OBJECT) 0.05`가 그 예이다. 엔티티 threshold가 0.2인데도 0.05인 엔티티가 남는 것은, 관계의 끝점으로 선택된 엔티티를 JointIE 디코더가 유지하기 때문으로 보인다.
- LLM도 완전하지 않다. `서비스별(OBJECT)`, 인용문 전체를 엔티티로 만들거나, `이재용`과 `이재용 회장` 중 무엇을 이름으로 쓸지가 회차마다 바뀐다.

### 관계 지표

GLiNER는 자신이 찾은 엔티티 사이에서만 관계를 디코딩하며, 스키마의 머리·꼬리 타입 제약을 디코딩 중에 적용한다. 관계 수는 반복당 평균이다.

In [ ]:
show_table(["모드", f"관계 적중 /{GOLD_RELATION_TOTAL}", "생성 관계 수"],
           [[mode, per_repeat_mean(mode, score_relations, "hit"), per_repeat_mean(mode, score_relations, "total")]
            for mode in MODES])

| 모드 | 관계 적중 /9 | 생성 관계 수 |
|---|---|---|
| base_whitespace | 2 | 5 |
| base_char | 0 | 0 |
| base_kiwi | 2 | 5 |
| multi_whitespace | 8* | 17 |
| multi_char | 1 | 4 |
| multi_kiwi | 8 | 16 |
| llm_only | 8 | 17.7 |

\* 조사가 붙은 이름의 포함 일치를 인정한 값이다.

**해석**

- GLiNER의 관계 품질은 엔티티 품질에 종속된다. 엔티티 범위가 틀리면 관계도 틀린 노드에 연결되므로, 체크포인트와 분할기에 따라 적중 수가 0~8로 갈린다.
- `multi_kiwi`는 적중 수만 보면 LLM과 같다(8 대 8). 차이는 적중 수가 아니라 **틀린 관계와 중복 관계**에서 드러난다.

### 문장별 관계

In [ ]:
REL_FOCUS = ["multi_whitespace", "multi_kiwi", "llm_only"]
show_table(["#", *REL_FOCUS],
           [[index, *(relation_summary(runs_of(mode, index)) for mode in REL_FOCUS)]
            for index in range(5, len(TEXTS) + 1)])

`multi_kiwi`가 생성한 관계 16개를 직접 판정하면 다음과 같다. confidence는 반복 1의 값이다(GLiNER는 반복 간 동일).

| 판정 | 관계 (confidence) |
|---|---|
| 정답 (8) | 삼성전자–엔비디아 PARTNER_WITH (0.91), 이재용 회장→삼성전자 EMPLOYED_BY (0.89), 김민수→AWS 서밋 PARTICIPATED_IN (0.84), AWS 서밋→부산 OCCURRED_AT (0.89), 네이버클라우드–KT PARTNER_WITH (0.92), 박지훈→토스 EMPLOYED_BY (0.73), 박지훈→성수동 RESIDES_AT (0.86), 쿠팡→서울 송파구 HEADQUARTERS_AT (0.69) |
| 허용 가능 (3) | 데이터센터→판교 LOCATED_AT (0.98), 네이버클라우드·KT→데이터센터 OWNS (0.77, 0.67) |
| 중복 타입 (3) | AWS 서밋→부산 LOCATED_AT (0.93, OCCURRED_AT과 중복), 박지훈→토스 MEMBER_OF (0.68, EMPLOYED_BY와 중복), 박지훈→성수동 LOCATED_AT (0.64, RESIDES_AT과 중복) |
| **문장에 없는 관계 (2)** | **김민수→카카오 EMPLOYED_BY (0.75)**: "카카오 개발자들과 함께"일 뿐 소속이 아님<br>**김민수→부산 RESIDES_AT (0.78)**: 행사 장소일 뿐 거주지가 아님 |

**해석**

- **틀린 관계의 confidence가 정답과 구분되지 않는다.** 틀린 두 관계(0.75, 0.78)는 정답인 `박지훈→토스 EMPLOYED_BY`(0.73), `쿠팡→서울 송파구 HEADQUARTERS_AT`(0.69)보다 높다. threshold로 걸러낼 수 없다.
- **GLiNER는 관계 타입을 서로 배타적으로 고르지 않는다.** 관계 타입마다 독립적으로 점수를 매기므로, 스키마 설명이 "더 구체적인 타입을 우선하라"고 한 `LOCATED_AT`/`RESIDES_AT`, `MEMBER_OF`/`EMPLOYED_BY`가 함께 나온다.
- **LLM은 문맥을 해석한다.** 같은 문장에서 LLM은 "카카오 개발자들과 함께"를 `카카오 -PARTICIPATED_IN→ AWS 서밋` 또는 `카카오 개발자들 -MEMBER_OF→ 카카오`로 읽었고, 거주 관계는 만들지 않았다. 대신 `PARTNER_WITH`를 2/3회, `EMPLOYED_BY`를 1/3회만 생성하는 등 회차별 변동이 크다.

## 결론과 한계

**결론**

1. 한국어에서 GLiNER의 가장 큰 문제인 조사 포함 범위는 모델이 아니라 단어 분할의 문제이며, 형태소 분할기로 해결된다. multi 체크포인트 + Kiwi 분할기 조합은 엔티티 정확 일치 16/21, 조사·타입 오류 0건으로 기본 설정(1/21)보다 크게 낫다.
2. 그래도 품질은 LLM에 미치지 못한다. 엔티티는 16 대 19.7이고, 관계는 적중 수가 같아도 문장에 없는 관계와 중복 타입이 섞이며 그 confidence가 정답과 겹친다.
3. GLiNER의 장점은 품질이 아니라 운영 특성이다. 문장당 약 0.14초로 LLM보다 약 13배 빠르고, 같은 입력에 항상 같은 결과를 내며, API 비용과 외부 전송이 없다.
4. 따라서 품질이 우선이면 LLM, 처리량·비용·결정성·데이터 반출 제약이 우선이면 multi + 형태소 분할기 조합이 현실적인 선택지이다. 두 방식을 결합한다면 결과를 합집합으로 병합하기보다 GLiNER로 엔티티를 찾고 LLM이 그 사이의 관계를 판단하는 분업 구조가 검토 대상이다(본 실험에서 검증하지 않음).

**한계**

- 문장 9개와 실험자가 정한 참고 정답에 기반한 진단이며 일반 벤치마크가 아니다. 정량 비교는 KLUE-NER 등 공개 한국어 데이터셋이 필요하다.
- 형태소 분할기는 공개 체크포인트의 학습 조건(공백 분할)과 다르다. 한국어 형태소 단위로 미세조정한 모델이라면 결과가 달라질 수 있다.
- LLM은 모델 하나(`gpt-5.4-mini`)와 하나의 프롬프트만 사용했다.